In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from src.to_html import buttons, cell, heading, paragraph, table

In [2]:
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[["delay_in_min", "station_name", "is_canceled", "train_type"]])
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 29,407,028 records from 2 files
Files: ['data-2025-11.parquet', 'data-2025-12.parquet']


In [3]:
def calculate_station_stats(df_filtered: pd.DataFrame) -> pd.DataFrame:
    """Calculate delay and cancellation statistics per station."""
    # Calculate average delays and stop counts for each station
    station_df = (
        df_filtered[~df_filtered["is_canceled"]]
        .groupby("station_name")["delay_in_min"]
        .agg(["mean", "count"])
        .reset_index()
        .sort_values("mean", ascending=False)
        .reset_index(drop=True)
    )
    station_df.columns = ["Bahnhof", "Durchschn. Verspätung [min]", "Anzahl Halte"]

    # Calculate cancellation rates for each station
    cancellation_df = (
        df_filtered.groupby("station_name").agg({"is_canceled": "mean"}).rename(columns={"is_canceled": "Ausfallquote"})
    )

    # Combine all statistics
    station_df = station_df.merge(cancellation_df, left_on="Bahnhof", right_index=True)

    # Format the columns
    station_df["Durchschn. Verspätung [min]"] = station_df["Durchschn. Verspätung [min]"].round(2)
    station_df["Ausfallquote"] = (station_df["Ausfallquote"] * 100).round(1).astype(str) + "%"
    station_df["Anzahl Halte"] = station_df["Anzahl Halte"].apply(lambda x: f"{x:,}")

    return station_df

In [4]:
# Calculate statistics for each train type
train_types = ["Alle", "ICE", "IC", "RE", "RB", "S"]
station_tables = {}

for train_type in train_types:
    if train_type == "Alle":
        df_filtered = df
    else:
        df_filtered = df[df["train_type"] == train_type]

    station_df = calculate_station_stats(df_filtered)
    station_tables[train_type] = table(station_df)

content = cell(
    heading("Wie ist die durchschnittliche Verspätung pro Bahnhof?", level=1)
    + paragraph(
        "Die Tabellen zeigen, wie die durchschnittlichen Verspätungen und Ausfallquote pro Bahnhof sind. "
        "Dabei ist zu beachten, dass die durchschnittliche Verspätung stark damit zusammenhängt, welche Zuggattungen an dem Bahnhof halten. "
        "Die Anzahl Halte gibt an, wie viele Zughalte es an dem Bahnhof gab und damit auch wie viele Zughalte zur Berechnung benutzt wurden."
    )
    + buttons(station_tables)
)
display(HTML(content))

Bahnhof ⇕,Durchschn. Verspätung [min] ⇕,Anzahl Halte ⇕,Ausfallquote ⇕
Ennepetal (Gevelsberg),11.96,"5,878",10.6%
Brühl,10.46,"5,522",3.9%
Erkelenz,10.29,"4,642",3.3%
Hückelhoven-Baal,10.29,"4,505",3.1%
Sinzig (Rhein),10.14,"4,598",3.0%
Bad Breisig,9.99,"4,595",3.0%
Altena (Westf),9.93,"4,210",2.2%
Frankfurt am Main Flughafen Fernbahnhof,9.81,"12,229",4.1%
Rolandseck,9.8,"2,489",2.1%
Willebadessen,9.79,"1,776",2.8%


In [5]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/verspaetung_pro_bahnhof.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))